# Badge 1: Data Warehousing Workshop: Lesson 7: The Load Data Wizard

VEGETABLE_DETAILS, loading two files, and the duplicate-Spinach cleanup.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `badge1_dora_checks.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `00_badge1_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH

In [ ]:
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🎭 Insert Statements Get Old, Fast

> **Course:** Create a Vegetable Details Table. Screenshot DWW_076 shows context **GARDEN_PLANTS.VEGGIES**; the role is not shown.

> ⚠️ **Audit note:** Screenshots disagree on the owner: DWW_088 (this lesson) shows VEGETABLE_DETAILS owned by SYSADMIN, DWW_099 (Lesson 8) shows ACCOUNTADMIN. This notebook follows SYSADMIN, the course's stated convention.

In [ ]:
use role {{SYSADMIN_ROLE}};
use schema GARDEN_PLANTS.VEGGIES;
create table garden_plants.veggies.vegetable_details
(
plant_name varchar(25)
, root_depth_code varchar(1)    
);

## Page: 🥋 Loading Table Rows from a File (A to K)

> **Course:** Download this file (veggie_details_a_to_k_comma_opt_enclosed.csv) and upload it into your Veggie Details table using the Load Data wizard (DWW_078–084): table → Load Data → choose file → File format **Delimited Files (CSV or TSV)** → View options → Header **First line contains header** → map *Plant Name* → PLANT_NAME and *Rooting Depth* → ROOT_DEPTH_CODE → **Load**.

> ⚠️ **Audit note:** The export's copy of the file is named `veggie_details_a_to_k_comm.csv` (Rise truncated the name); the contents are the same file.

In [ ]:
-- SQL equivalent of the Load Data wizard: the wizard reads the learner's downloaded file directly
copy into GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('veggie_details_a_to_k_comm.csv')
file_format = (type = 'CSV' field_delimiter = ',' skip_header = 1 field_optionally_enclosed_by = '"');

## Page: 🥋 Loading Table Rows from a File (K to Z)

> **Course:** 🎯 Challenge Lab: Load the second file (veggie_details_k_to_z_pipe.csv) into the same table using the same method. The columns are NOT separated by commas this time.

In [ ]:
-- Derived solution: same wizard settings, Field delimiter changed to |
copy into GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('veggie_details_k_to_z_pipe.csv')
file_format = (type = 'CSV' field_delimiter = '|' skip_header = 1 field_optionally_enclosed_by = '"');

> **Course:** Check your table by running a select statement. You should see 42 rows. If you reverse the sort order, you should see Zucchini at the top.

In [ ]:
select * from GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS order by plant_name desc;

In [ ]:
print('Rows:', len(check_42_rows), '(course expects 42)'); print('Top row:', check_42_rows['PLANT_NAME'][0])

The course's reset statement, if a file was loaded twice (not run):

```sql
TRUNCATE TABLE GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS;
```

### 🤖 DORA check DWW06 goes here
The course has the learner run **DWW06** (VEGETABLE_DETAILS Table) at this point.
It is in `badge1_dora_checks.ipynb`, section **DWW06**, as a local check plus the exact DORA code.

## Page: 🎯 Viewing Our Table Data

> **Course:** View your data · Notice a plant name that appears twice · Find a way to get rid of it · View your data again (DWW_090–094).

In [ ]:
select * from GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS;

In [ ]:
-- Derived from DWW_091
select * from GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS where plant_name = 'Spinach';

In [ ]:
-- Derived from DWW_092
select * from GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS where plant_name = 'Spinach' and root_depth_code = 'D';

In [ ]:
-- From DWW_093
use schema GARDEN_PLANTS.VEGGIES;
delete from vegetable_details
where plant_name = 'Spinach'
and root_depth_code = 'D';

In [ ]:
-- Derived from DWW_094
select plant_name, count(*) as n from GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS group by plant_name having count(*) > 1;

### 🤖 DORA check DWW07 goes here
The course has the learner run **DWW07** (VEG_DETAILS row count, expects 41) at this point.
It is in `badge1_dora_checks.ipynb`, section **DWW07**, as a local check plus the exact DORA code.